# G17 — Round 2 qualitative boundary audit (Ahorts-style)
Best Round-2 models vs the Exp-16 loss. Native-resolution contours on the **266-scan test set**: ground truth (**green**), baseline BCE+Dice / Exp-16 loss (**orange, dashed**) and each candidate loss (solid).
Each panel: Dice / 95HD / ASD / 2-px Boundary F1 for both models (flip-TTA, threshold 0.5). Cases are picked by rule (largest gains, largest regressions, all-model failures), not by eye.
Needs the **G17_loss_search_R2** output attached as Input; CPU is enough.

In [ ]:
!pip -q install datasets

In [ ]:
%%writefile report_core.py
# G17 — Ahorts-style qualitative boundary audit for a finished search round.
# Native-resolution contours: ground truth (green) vs baseline (orange) vs candidate (cyan/magenta),
# per-case Dice / 95HD / ASD / 2-px Boundary F1, plus summary statistics with paired tests.
import glob, json, os, sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

GT_C, BASE_C = "#39d353", "#ff9f1c"
CAND_C = {"boundary": "#2ec4f1", "ft": "#ff4fd8", "ft4060": "#b28dff"}


def find_runs_root():
    c = sorted(glob.glob("/kaggle/input/**/runs", recursive=True)) + ["/kaggle/working/runs", "runs"]
    for d in c:
        if os.path.isdir(d):
            return d
    raise FileNotFoundError("no runs/ folder found — attach the search notebook's output as Input")


def load_run(root, name, variant="tta"):
    d = Path(root) / name
    df = pd.read_csv(d / f"test_{variant}.csv").set_index("image_id")
    probs = np.load(d / "test_probs.npz")["tta" if "tta" in variant else "plain"]
    return df, probs


def native_contour_mask(prob_u8, shape):
    p = cv2.resize(prob_u8.astype(np.float32) / 255.0, (shape[1], shape[0]), interpolation=cv2.INTER_LINEAR)
    return p >= 0.5


def crop_box(masks, shape, pad=0.35, min_side=96):
    m = np.zeros(shape, bool)
    for k in masks:
        m |= k
    if not m.any():
        return 0, shape[0], 0, shape[1]
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    h, w = max(y1 - y0, min_side), max(x1 - x0, min_side)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    s = max(h, w) * (1 + pad)
    y0, y1 = int(max(0, cy - s / 2)), int(min(shape[0], cy + s / 2))
    x0, x1 = int(max(0, cx - s / 2)), int(min(shape[1], cx + s / 2))
    return y0, y1, x0, x1


def draw_case(ax, img, gt, preds, title):
    """preds: list of (mask, colour, linestyle)."""
    y0, y1, x0, x1 = crop_box([gt] + [p[0] for p in preds], gt.shape)
    ax.imshow(img[y0:y1, x0:x1], cmap="gray", interpolation="lanczos")
    ax.contour(gt[y0:y1, x0:x1], levels=[0.5], colors=GT_C, linewidths=1.8)
    for m, c, ls in preds:
        if m[y0:y1, x0:x1].any():
            ax.contour(m[y0:y1, x0:x1], levels=[0.5], colors=c, linewidths=1.4, linestyles=ls)
    # full-frame inset so distant false positives (outside the crop) stay visible
    ins = ax.inset_axes([0.70, 0.70, 0.29, 0.29])
    ins.imshow(img, cmap="gray"); ins.axis("off")
    ins.contour(gt, levels=[0.5], colors=GT_C, linewidths=0.6)
    for m, c, _ in preds:
        if m.any():
            ins.contour(m, levels=[0.5], colors=c, linewidths=0.6)
    ins.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="white", lw=0.6))
    ax.set_title(title, fontsize=7.2, loc="left")
    ax.axis("off")


def fmt(row):
    return f"D {row.dice*100:.1f} | 95HD {row.hd95:.1f} | ASD {row.asd:.2f} | BF1 {row.bf1*100:.0f}"


def page(pdf, cases, native, base, cand, title, subtitle, takeaway, png_path, cand_colour, base_label, cand_label):
    n = len(cases)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig = plt.figure(figsize=(11.7, 3.25 * rows + 2.0))
    fig.suptitle(title, fontsize=13, fontweight="bold", x=0.02, ha="left", y=0.995)
    fig.text(0.02, 1 - 0.55 / (3.25 * rows + 2.0), subtitle, fontsize=8.5, color="#555")
    for k, cid in enumerate(cases):
        ax = fig.add_subplot(rows, cols, k + 1)
        img, gt = native[cid]
        b_mask = native_contour_mask(base["probs"][base["idx"][cid]], gt.shape)
        c_mask = native_contour_mask(cand["probs"][cand["idx"][cid]], gt.shape)
        b, c = base["df"].loc[cid], cand["df"].loc[cid]
        t = (f"{cid}  [{b.dataset}]\n{base_label}: {fmt(b)}\n{cand_label}: {fmt(c)}")
        draw_case(ax, img, gt, [(b_mask, BASE_C, "--"), (c_mask, cand_colour, "-")], t)
    handles = [Line2D([], [], color=GT_C, lw=2, label="Ground truth"),
               Line2D([], [], color=BASE_C, lw=2, ls="--", label=base_label),
               Line2D([], [], color=cand_colour, lw=2, label=cand_label)]
    fig.legend(handles=handles, loc="lower left", ncol=3, fontsize=8.5, frameon=False, bbox_to_anchor=(0.02, 0.0))
    fig.text(0.02, 0.035, "Takeaway: " + takeaway, fontsize=8.3, wrap=True, va="bottom")
    plt.tight_layout(rect=(0, 0.07, 1, 0.96))
    fig.savefig(png_path, dpi=130); pdf.savefig(fig); plt.close(fig)


def build_report(runs_root, split_test, out_dir, models, base_key, title_prefix="Round 1"):
    """models: {key: (run_name, label, colour)}; base_key is the baseline key."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    M = {}
    for k, (run, label, colour) in models.items():
        df, probs = load_run(runs_root, run)
        ids = list(pd.read_csv(Path(runs_root) / run / "test_plain.csv").image_id)
        M[k] = dict(run=run, label=label, colour=colour, df=df, probs=probs, idx={c: i for i, c in enumerate(ids)})
    # native-resolution images + masks from the dataset
    native = {}
    for r in split_test:
        native[r["image_id"]] = (np.asarray(r["raw_image"].convert("L")), np.asarray(r["mask"].convert("L")) > 127)
    base = M[base_key]
    common = [c for c in base["df"].index if c in native]
    stats_rows, pages = [], []
    pdf = PdfPages(out_dir / "G17_visual_report.pdf")

    # ---- summary page ----
    from scipy.stats import wilcoxon
    fig, axes = plt.subplots(2, 2, figsize=(11.7, 8.3))
    fig.suptitle(f"{title_prefix} — summary on the 266-scan test set (flip-TTA, threshold 0.5)", fontsize=13,
                 fontweight="bold", x=0.02, ha="left")
    rows = []
    for k, m in M.items():
        d = m["df"].loc[common]
        r = {"model": m["label"], "Dice %": d.dice.mean() * 100, "IoU %": d.jaccard.mean() * 100,
             "95HD px": d.hd95.mean(), "ASD px": d.asd.mean(), "BF1 %": d.bf1.mean() * 100,
             "Dice<50 %": (d.dice < 0.5).mean() * 100}
        for coh in ["BUSI", "BUS_BRA", "BUS_UCLM"]:
            r[f"Dice {coh}"] = d[d.dataset == coh].dice.mean() * 100
        if k != base_key:
            b = base["df"].loc[common]
            r["wins vs base"] = int((d.dice > b.dice + 1e-4).sum()); r["losses"] = int((d.dice < b.dice - 1e-4).sum())
            r["p Dice"] = wilcoxon(d.dice, b.dice).pvalue; r["p BF1"] = wilcoxon(d.bf1, b.bf1).pvalue
            r["p 95HD"] = wilcoxon(d.hd95, b.hd95).pvalue
        rows.append(r)
    S = pd.DataFrame(rows); S.to_csv(out_dir / "summary_table.csv", index=False)
    ax = axes[0, 0]; ax.axis("off")
    show = S[["model", "Dice %", "95HD px", "ASD px", "BF1 %", "Dice<50 %"]].round(2)
    tb = ax.table(cellText=show.values, colLabels=show.columns, loc="center", cellLoc="center",
                  colWidths=[0.40] + [0.12] * (show.shape[1] - 1))
    tb.auto_set_font_size(False); tb.set_fontsize(7.5); tb.scale(1, 1.5); ax.set_title("Overall (test, TTA)", fontsize=9)
    ax = axes[0, 1]; ax.axis("off")
    cols = ["model"] + [c for c in ["wins vs base", "losses", "p Dice", "p BF1", "p 95HD"] if c in S]
    show2 = S[cols].copy()
    for c in ["p Dice", "p BF1", "p 95HD"]:
        if c in show2:
            show2[c] = show2[c].map(lambda v: "" if pd.isna(v) else f"{v:.3f}")
    tb = ax.table(cellText=show2.fillna("").values, colLabels=show2.columns, loc="center", cellLoc="center",
                  colWidths=[0.40] + [0.12] * (show2.shape[1] - 1))
    tb.auto_set_font_size(False); tb.set_fontsize(7.5); tb.scale(1, 1.5)
    ax.set_title(f"Paired vs {base['label']} (Wilcoxon signed-rank, per case)", fontsize=9)
    ax = axes[1, 0]
    data = [M[k]["df"].loc[common].dice * 100 for k in M]
    ax.boxplot(data, tick_labels=[M[k]["label"].replace(" ", "\n", 1) for k in M], showfliers=True, flierprops=dict(ms=2))
    ax.set_ylabel("Dice % per case"); ax.tick_params(labelsize=7); ax.set_title("Per-case Dice", fontsize=9)
    ax = axes[1, 1]
    data = [M[k]["df"].loc[common].bf1 * 100 for k in M]
    ax.boxplot(data, tick_labels=[M[k]["label"].replace(" ", "\n", 1) for k in M], showfliers=True, flierprops=dict(ms=2))
    ax.set_ylabel("2-px Boundary F1 % per case"); ax.tick_params(labelsize=7); ax.set_title("Per-case Boundary F1", fontsize=9)
    plt.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(out_dir / "report_p0_summary.png", dpi=130); pdf.savefig(fig); plt.close(fig)
    pages.append("report_p0_summary.png")

    # ---- qualitative pages (cases chosen by rule, not by eye) ----
    b = base["df"].loc[common]
    pn = 1
    for k, m in M.items():
        if k == base_key:
            continue
        c = m["df"].loc[common]
        ok = (b.dice > 0.5) & (c.dice > 0.5)
        gain_bf1 = (c.bf1 - b.bf1)[ok].sort_values(ascending=False)
        gain_hd = (b.hd95 - c.hd95).sort_values(ascending=False)
        loss_hd = (c.hd95 - b.hd95).sort_values(ascending=False)
        sel = [
            ("Margin refinement", list(gain_bf1.index[:6]),
             f"Largest 2-px Boundary-F1 gains of {m['label']} over {base['label']} (both Dice > 50%).",
             f"mean BF1 change on the whole test set {(c.bf1 - b.bf1).mean()*100:+.1f} pts; "
             f"{int(((c.bf1 - b.bf1) > 0.05).sum())} cases gain > 5 pts, {int(((c.bf1 - b.bf1) < -0.05).sum())} lose > 5 pts."),
            ("Outlier / shadow clean-up", list(gain_hd.index[:6]),
             f"Largest 95HD reductions of {m['label']} vs {base['label']} (distant false-positive islands removed).",
             f"median 95HD change {(c.hd95 - b.hd95).median():+.2f} px; "
             f"{int(((b.hd95 - c.hd95) > 20).sum())} cases improve by > 20 px, {int(((c.hd95 - b.hd95) > 20).sum())} get worse by > 20 px."),
            ("Regressions (honest view)", list(loss_hd.index[:6]),
             f"Cases where {m['label']} is WORSE than {base['label']} (largest 95HD increases).",
             "these are the failure modes to discuss: most are low-contrast or multi-lobed lesions where "
             "both models disagree with the annotation; check the inset for distant islands."),
        ]
        for title, cases, subtitle, takeaway in sel:
            if not cases:
                continue
            png = f"report_p{pn}_{k}_{title.split()[0].lower()}.png"
            page(pdf, cases, native, base, m, f"{title}: {m['label']} vs {base['label']}", subtitle, takeaway,
                 out_dir / png, m["colour"], base["label"], m["label"])
            pages.append(png); pn += 1
    # ---- all-model hard cases ----
    allfail = [cid for cid in common if all(M[k]["df"].loc[cid].dice < 0.5 for k in M)]
    if allfail:
        cases = allfail[:6]
        fig, axes = plt.subplots(int(np.ceil(len(cases) / 3)), 3, figsize=(11.7, 3.4 * int(np.ceil(len(cases) / 3)) + 1.2), squeeze=False)
        fig.suptitle(f"Hard cases — every model below 50% Dice ({len(allfail)} of {len(common)} scans)", fontsize=13,
                     fontweight="bold", x=0.02, ha="left")
        for ax in axes.ravel():
            ax.axis("off")
        for j, cid in enumerate(cases):
            img, gt = native[cid]
            preds = [(native_contour_mask(M[k]["probs"][M[k]["idx"][cid]], gt.shape), M[k]["colour"], "-" if k != base_key else "--") for k in M]
            draw_case(axes.ravel()[j], img, gt, preds, f"{cid} [{base['df'].loc[cid].dataset}]  " +
                      "  ".join(f"{M[k]['label'][:10]} D{M[k]['df'].loc[cid].dice*100:.0f}" for k in M))
        handles = [Line2D([], [], color=GT_C, lw=2, label="Ground truth")] + \
                  [Line2D([], [], color=M[k]["colour"], lw=2, ls="--" if k == base_key else "-", label=M[k]["label"]) for k in M]
        fig.legend(handles=handles, loc="lower left", ncol=len(handles), fontsize=8, frameon=False)
        plt.tight_layout(rect=(0, 0.05, 1, 0.95))
        png = f"report_p{pn}_hard_cases.png"; fig.savefig(out_dir / png, dpi=130); pdf.savefig(fig); plt.close(fig)
        pages.append(png)
    pdf.close()
    json.dump({"pages": pages, "n_cases": len(common), "all_fail": allfail}, open(out_dir / "report_index.json", "w"), indent=1)
    print(S.round(3).to_string(index=False))
    return pages


In [ ]:
import glob, shutil, sys, os
src = sorted(glob.glob('/kaggle/input/**/g17core.py', recursive=True))[0]; shutil.copy(src, 'g17core.py')
sys.path.insert(0, '.'); import g17core as g, report_core as R
split = g.load_splits()
root = R.find_runs_root(); print('runs from', root, sorted(os.listdir(root)))
MODELS = {"base": ("R1_dice_repro", "BCE+Dice (Exp-16 loss)", R.BASE_C),
          "boundary": ("ENS_all9", "Ensemble of 9 (FT ×6 + boundary ×3)", R.CAND_C["boundary"]),
          "ft": ("R2_ft55_morph05", "Focal Tversky α0.5/β0.5 + boundary λ0.5", R.CAND_C["ft"]),
          "ft4060": ("ENS_ft4060_x3", "Ensemble FT α0.4/β0.6 ×3", R.CAND_C["ft4060"])}
pages = R.build_report(root, split["test"], "report", MODELS, "base", title_prefix="Round 2")

In [ ]:
from IPython.display import Image, display
for p in pages:
    print(p); display(Image(os.path.join('report', p)))